# Clinical Problem and Objective

**The Clinical Problem:** Post-myocardial infarction management requires predicting severe complications (e.g., arrhythmias, heart failure, ruptures) rather than simply binary survival. These complications are rare, resulting in extreme class imbalance. This frequently causes traditional machine learning models to default to predicting the majority class ("no complication") to artificially inflate accuracy
a phenomenon known as the Accuracy Paradox.

**Project Objective:** 
* Build a high-sensitivity predictive system optimized for **Recall**.
* Accurately forecast 12 distinct clinical targets simultaneously.
* Handle a "small data" regime without severe overfitting.

In [8]:
import warnings
warnings.filterwarnings('ignore')
!pip install autogluon.tabular ucimlrepo google-adk

# Dataset Overview

**Source:** *Myocardial Infarction Complications Data Set* (UCI Machine Learning Repository).

**Data Dimensions:**
* **Instances:** 1,700 patients.
* **Input Variables (Features):** 111 (Demographics, Anamnesis, ECG findings, Biomarkers, Vitals).
* **Targets (12 Outcomes):** 11 Binary Complications and 1 Multiclass Target (Lethal Outcome).

**Sparsity and Missing Data:**
The dataset exhibits severe Missing Not At Random (MNAR) patterns, representing real-world clinical triage. Extreme variables reach >60% missingness (e.g., Family history, Systolic BP in the ER). Aggressive imputation can destroy the predictive signal inherent in the "absence" of a clinical test.

In [9]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
import xgboost as xgb
from ucimlrepo import fetch_ucirepo
from autogluon.tabular import TabularDataset, TabularPredictor

# 1. Fetch data from the UCI repository
print("Fetching dataset...")
mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()
variables_info = mi_data.variables

# 2. Identify Column Types
cat_cols_info = variables_info[(variables_info['role'] == 'Feature') & (variables_info['type'] == 'Categorical')]['name'].tolist()
categorical_cols = [c for c in cat_cols_info if c in X_full.columns]
numeric_cols = [c for c in X_full.columns if c not in categorical_cols]

# 3. Fill missing targets with the most frequent value (mode)
y_full = y_full.fillna(y_full.mode().iloc[0])

target_names = y_full.columns.tolist()
binary_targets = target_names[:-1]
multiclass_target = target_names[-1]

print(f"Loaded structure: {X_full.shape[0]} patients, {X_full.shape[1]} features, {y_full.shape[1]} targets.")

Fetching dataset...
Loaded structure: 1700 patients, 111 features, 12 targets.


# The Temporal Challenge and Data Masking

Clinical information is unlocked chronologically over 72 hours. A static model risks **Data Leakage** (using future treatments to predict present risks).

**Masking Strategy:**
We create distinct datasets based on the clinical timeline. When predicting at *Admission*, variables from Days 1, 2, and 3 are strictly masked to ensure valid real-time simulations. The Temporal Augmentation function concatenates these stages to maximize the sample size.

In [10]:
def generate_temporal_datasets(X_data):
    """
    Generates four distinct datasets based on the clinical timeline.
    Removes future columns to prevent data leakage.
    """
    day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
    day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
    day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']

    datasets = {}
    drop_admission = day_1_cols + day_2_cols + day_3_cols
    datasets['admission'] = X_data.drop(columns=[c for c in drop_admission if c in X_data.columns])

    drop_day_1 = day_2_cols + day_3_cols
    datasets['day_1'] = X_data.drop(columns=[c for c in drop_day_1 if c in X_data.columns])

    drop_day_2 = day_3_cols
    datasets['day_2'] = X_data.drop(columns=[c for c in drop_day_2 if c in X_data.columns])

    datasets['day_3'] = X_data.copy()
    return datasets

def create_augmented_dataset(X_base, y_base):
    """
    Concatenates the 4 timelines into a single dataset.
    Missing future variables are naturally handled as NaNs.
    """
    temporal_dicts = generate_temporal_datasets(X_base)
    X_list, y_list = [], []
    stages = {'admission': 0, 'day_1': 1, 'day_2': 2, 'day_3': 3}

    for stage_name, df_stage in temporal_dicts.items():
        df_stage = df_stage.copy()
        df_stage['TIMELINE_STAGE'] = stages[stage_name]
        X_list.append(df_stage)
        y_list.append(y_base.copy())

    X_aug = pd.concat(X_list, ignore_index=True)
    y_aug = pd.concat(y_list, ignore_index=True)
    return X_aug, y_aug

def fix_categorical_types(df, cat_columns):
    for col in cat_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).replace({'nan': 'Unknown', 'NaN': 'Unknown'}).astype('category')
    df['TIMELINE_STAGE'] = df['TIMELINE_STAGE'].astype('category')
    return df
    
# Train/Test Split (Performed before augmentation to prevent leakage)
X_train_temp, X_test_base, y_train_temp, y_test_base = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

X_train_base, X_val_base, y_train_base, y_val_base = train_test_split(
    X_train_temp, y_train_temp, test_size=0.2, random_state=42
)

print("Executing Temporal Augmentation on the training set...")
X_train_aug, y_train_aug = create_augmented_dataset(X_train_base, y_train_base)
X_val_aug, y_val_aug = create_augmented_dataset(X_val_base, y_val_base)

Executing Temporal Augmentation on the training set...


In [11]:
from sklearn.model_selection import train_test_split
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------------------------------------------------------
# 1. Data Processing and Validation Split
# ---------------------------------------------------------
cat_cols_nn = categorical_cols.copy() + ['TIMELINE_STAGE']
num_cols_nn = numeric_cols.copy()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols_nn),
        ('cat', categorical_transformer, cat_cols_nn)
    ])

X_train_processed = preprocessor.fit_transform(X_train_aug)
X_val_processed = preprocessor.transform(X_val_aug)

# Targets Train
y_train_bin = y_train_aug[binary_targets].values.astype(np.float32)
y_train_multi = y_train_aug[multiclass_target].values.astype(np.int64)

# Targets Validation
y_val_bin = y_val_aug[binary_targets].values.astype(np.float32)
y_val_multi = y_val_aug[multiclass_target].values.astype(np.int64)

# Train Tensors
X_t_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
y_b_t_tensor = torch.tensor(y_train_bin, dtype=torch.float32)
y_m_t_tensor = torch.tensor(y_train_multi, dtype=torch.long)

# Validation Tensors
X_v_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
y_b_v_tensor = torch.tensor(y_val_bin, dtype=torch.float32)
y_m_v_tensor = torch.tensor(y_val_multi, dtype=torch.long)

train_dataset = TensorDataset(X_t_tensor, y_b_t_tensor, y_m_t_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset = TensorDataset(X_v_tensor, y_b_v_tensor, y_m_v_tensor)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print("Processing done")

Processing done


In [12]:
!pip install gcastle networkx matplotlib

In [13]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler # <-- IMPORTANTE

from castle.algorithms import Notears

# ---------------------------------------------------------
# 1. DEFINIÇÃO DAS COLUNAS E FATIAS DE TEMPO
# ---------------------------------------------------------
day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
targets = binary_targets + [multiclass_target]

future_cols = day_1_cols + day_2_cols + day_3_cols + targets

# Puxando os dados COMPLETOS apenas para os pacientes de treino
X_train_full = X_full.loc[X_train_base.index].copy()

adm_cols = [c for c in X_train_full.columns if c not in future_cols]

tier_0 = adm_cols
tier_1 = day_1_cols
tier_2 = day_2_cols
tier_3 = day_3_cols + targets

# Preparando o DataFrame causal completo
df_causal = pd.concat([X_train_full, y_train_base], axis=1)
df_causal = df_causal.apply(pd.to_numeric, errors='coerce').fillna(0)
col_names = df_causal.columns.tolist()

# ---> CORREÇÃO CRÍTICA DE PERFORMANCE <---
# Padronizando os dados matematicamente para o gradiente não travar
scaler = StandardScaler()
df_causal_scaled = pd.DataFrame(
    scaler.fit_transform(df_causal), 
    columns=col_names, 
    index=df_causal.index
)

# ---------------------------------------------------------
# 2. TREINAMENTO DO ALGORITMO CAUSAL (NOTEARS)
# ---------------------------------------------------------
print("Aprendendo o Grafo Causal com dados Normalizados...")

# Adicionamos max_iter=50 para forçar uma parada caso ele fique muito perfeccionista
nt = Notears(w_threshold=0.05, max_iter=50)

# Passamos a base normalizada!
nt.learn(df_causal_scaled)

adj_matrix = nt.causal_matrix.copy()

# ---------------------------------------------------------
# 3. APLICAÇÃO DA MÁSCARA TEMPORAL (PÓS-PROCESSAMENTO)
# ---------------------------------------------------------
def ban_edge(source, target):
    """Se o algoritmo achou que o futuro causou o passado, nós apagamos a seta"""
    if source in col_names and target in col_names:
        i = col_names.index(source)
        j = col_names.index(target)
        adj_matrix[i, j] = 0

# Aplicando as regras cronológicas rigorosamente na matriz
for c1 in tier_1:
    for c0 in tier_0:
        ban_edge(c1, c0)

for c2 in tier_2:
    for c0 in tier_0 + tier_1:
        ban_edge(c2, c0)

for c3 in tier_3:
    for c0 in tier_0 + tier_1 + tier_2:
        ban_edge(c3, c0)

# ---------------------------------------------------------
# 4. RECONSTRUÇÃO DO GRAFO E VISUALIZAÇÃO
# ---------------------------------------------------------
sm = nx.DiGraph()

for i, col_i in enumerate(col_names):
    sm.add_node(col_i)
    for j, col_j in enumerate(col_names):
        if adj_matrix[i, j] == 1:
            sm.add_edge(col_i, col_j)

print(f"Grafo Causal Limpo e Construído! Total de conexões (setas biológicas): {len(sm.edges())}")

plt.figure(figsize=(20, 15))
nx.draw_networkx(
    sm, 
    with_labels=True, 
    node_size=400, 
    node_color="lightblue", 
    font_size=7,
    alpha=0.8,
    arrows=True
)
plt.title("DAG Temporal - Cascata Fisiológica (gcastle - Notears com Restrição)")
plt.savefig("causal_dag_gcastle.png", dpi=300, bbox_inches='tight')
plt.show()

# ---------------------------------------------------------
# 5. EXTRAÇÃO DO MARKOV BLANKET (FILTRO CAUSAL)
# ---------------------------------------------------------
causal_parents_dict = {}
for target in targets:
    # Captura APENAS os pais que pertencem aos dados de Admissão
    # Isso blindará o TabPFN de qualquer vazamento de dados no Bloco 5
    parents = [edge[0] for edge in sm.edges() if edge[1] == target and edge[0] in adm_cols]
    causal_parents_dict[target] = parents

2026-05-29 02:31:45,415 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/linear.py[line:195] - INFO: [start]: n=1088, d=123, iter_=50, h_=1e-08, rho_=1e+16


Aprendendo o Grafo Causal com dados Normalizados...


2026-05-29 02:31:48,794 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/linear.py[line:206] - INFO: [iter 0] h=7.184e-01, loss=61.000, rho=1.0e+00


KeyboardInterrupt: 

In [21]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# Importamos o GOLEM (Sucessor do Notears na mesma biblioteca)
from castle.algorithms import GOLEM

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler # <-- IMPORTANTE

from castle.algorithms import Notears

# ---------------------------------------------------------
# 1. DEFINIÇÃO DAS COLUNAS E FATIAS DE TEMPO
# ---------------------------------------------------------
day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
targets = binary_targets + [multiclass_target]

future_cols = day_1_cols + day_2_cols + day_3_cols + targets

# Puxando os dados COMPLETOS apenas para os pacientes de treino
X_train_full = X_full.loc[X_train_base.index].copy()

adm_cols = [c for c in X_train_full.columns if c not in future_cols]

tier_0 = adm_cols
tier_1 = day_1_cols
tier_2 = day_2_cols
tier_3 = day_3_cols + targets

# Preparando o DataFrame causal completo
df_causal = pd.concat([X_train_full, y_train_base], axis=1)
df_causal = df_causal.apply(pd.to_numeric, errors='coerce').fillna(0)
col_names = df_causal.columns.tolist()

# ---> CORREÇÃO CRÍTICA DE PERFORMANCE <---
# Padronizando os dados matematicamente para o gradiente não travar
scaler = StandardScaler()
df_causal_scaled = pd.DataFrame(
    scaler.fit_transform(df_causal), 
    columns=col_names, 
    index=df_causal.index
)
# ---------------------------------------------------------
# 2. TREINAMENTO DO ALGORITMO CAUSAL (GOLEM)
# ---------------------------------------------------------
print("Aprendendo o Grafo Causal denso com GOLEM...")

# lambda_1 controla a punição (sparsity). Um valor baixo (2e-3) garante muitas conexões lógicas.
print("Aprendendo o Grafo Causal denso com GOLEM...")

# A SOLUÇÃO: graph_thres=0.01
# Qualquer conexão com força superior a 1% não será apagada pelo filtro nativo.
gl = GOLEM(lambda_1=2e-3, lambda_2=5.0, num_iter=20000, graph_thres=0.01)
gl.learn(df_causal_scaled)

# Pegamos a matriz binarizada, que agora estará muito mais rica e interconectada
adj_matrix = gl.causal_matrix.copy()

# ---------------------------------------------------------
# 3. APLICAÇÃO DA MÁSCARA TEMPORAL (PÓS-PROCESSAMENTO)
# ---------------------------------------------------------
def ban_edge(source, target):
    """Se o algoritmo achou que o futuro causou o passado, nós apagamos a seta"""
    if source in col_names and target in col_names:
        i = col_names.index(source)
        j = col_names.index(target)
        adj_matrix[i, j] = 0

for c1 in tier_1:
    for c0 in tier_0: ban_edge(c1, c0)
for c2 in tier_2:
    for c0 in tier_0 + tier_1: ban_edge(c2, c0)
for c3 in tier_3:
    for c0 in tier_0 + tier_1 + tier_2: ban_edge(c3, c0)

2026-05-29 03:00:22,184 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/torch/golem.py[line:119] - INFO: GPU is available.
2026-05-29 03:00:22,190 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/torch/golem.py[line:190] - INFO: Started training for 20000 iterations.
2026-05-29 03:00:22,193 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/torch/golem.py[line:203] - INFO: [Iter 0] score=725.461, likelihood=725.461, h=0.0e+00


Aprendendo o Grafo Causal denso com GOLEM...
Aprendendo o Grafo Causal denso com GOLEM...


2026-05-29 03:01:19,602 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/torch/golem.py[line:203] - INFO: [Iter 5000] score=709.978, likelihood=709.238, h=7.1e-03
2026-05-29 03:02:19,523 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/torch/golem.py[line:203] - INFO: [Iter 10000] score=709.958, likelihood=709.236, h=6.8e-03
2026-05-29 03:03:19,043 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/torch/golem.py[line:203] - INFO: [Iter 15000] score=709.958, likelihood=709.237, h=6.8e-03
2026-05-29 03:04:18,249 - /usr/local/lib/python3.12/dist-packages/castle/algorithms/gradient/notears/torch/golem.py[line:203] - INFO: [Iter 20000] score=709.958, likelihood=709.237, h=6.8e-03


In [24]:
print(sum(adj_matrix))

[2 0 0 0 6 0 0 0 4 4 0 0 0 0 0 4 1 2 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 1 1 0 6
 0 0 4 0 0 2 3 1 0 0 0 0 0 2 1 4 5 0 0 0 0 4 4 5 0 1 0 0 0 0 5 0 0 0 0 0 0
 0 0 0 0 0 2 0 0 2 0 0 3 0 0 2 0 0 4 1 2 0 0 0 0 0 7 2 1 0 6 0 0 0 2 1 0 0
 0 0 0 0 0 0 0 0 0 2 0 0]


In [20]:
weight_matrix = gl.weight_matrix if hasattr(gl, 'weight_matrix') else gl._B.detach().cpu().numpy()

# ---------------------------------------------------------
# 3. APLICAÇÃO DA MÁSCARA TEMPORAL (PÓS-PROCESSAMENTO)
# ---------------------------------------------------------
def ban_edge(source, target):
    """Zera a força gravitacional/matemática de eventos futuros afetando o passado"""
    if source in col_names and target in col_names:
        i = col_names.index(source)
        j = col_names.index(target)
        weight_matrix[i, j] = 0.0

for c1 in tier_1:
    for c0 in tier_0: ban_edge(c1, c0)
for c2 in tier_2:
    for c0 in tier_0 + tier_1: ban_edge(c2, c0)
for c3 in tier_3:
    for c0 in tier_0 + tier_1 + tier_2: ban_edge(c3, c0)

# ---------------------------------------------------------
# 4. EXTRAÇÃO DO MARKOV BLANKET RANKIADO (FILTRO CAUSAL)
# ---------------------------------------------------------
causal_parents_dict = {}

# Consideramos apenas conexões que possuem um ruído matemático maior que 0.01
MIN_WEIGHT = 0.01 

for i, target in enumerate(targets):
    if target in col_names:
        target_idx = col_names.index(target)
        
        parent_weights = []
        for j, col_j in enumerate(col_names):
            # Lê a força da seta da variável clínica para a doença
            weight = abs(weight_matrix[j, target_idx])
            
            # Garante que é forte o suficiente e pertence estritamente ao Dia 0
            if weight > MIN_WEIGHT and col_j in adm_cols:
                parent_weights.append((col_j, weight))
        
        # Ordena da causa biológica mais forte para a mais fraca
        parent_weights.sort(key=lambda x: x[1], reverse=True)
        
        # Pega as 10 conexões mais fortes para a doença.
        # Isso dá informação rica ao TabPFN, mas sem afogá-lo nas 111 variáveis puras
        top_causal_features = [pw[0] for pw in parent_weights[:10]]
        causal_parents_dict[target] = top_causal_features
    else:
        causal_parents_dict[target] = []

print("\n--- Exemplo de Features Rankeadas ---")
for t in binary_targets[:3]: 
    print(f"{t}: {causal_parents_dict[t]}")

AttributeError: 'GOLEM' object has no attribute '_B'

In [ ]:
# Salva a matriz de adjacência (o mapa completo do grafo)
df_adj = pd.DataFrame(adj_matrix, index=col_names, columns=col_names)
df_adj.to_csv("causal_adj_matrix.csv")

print("Matriz de adjacência salva em: causal_adj_matrix.csv")

In [ ]:
!pip uninstall "tabpfn @ git+https://github.com/PriorLabs/TabPFN.git" -y

In [ ]:
# Limpa pacotes conflitantes
!pip uninstall -y tabpfn-client autogluon.tabular autogluon.common autogluon.core autogluon.features
# Instala versões estáveis e o TabPFN local
!pip install pandas==2.2.2 pyarrow==17.0.0
!pip install "tabpfn @ git+https://github.com/PriorLabs/TabPFN.git"

In [ ]:
!pip install pandas==2.2.2 pyarrow==14.0.1 requests==2.32.4 jupyter-server==2.14.0

In [ ]:
!pip install tabpfn_client

In [ ]:
import tabpfn_client
from tabpfn_client import TabPFNClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix, fbeta_score
from ucimlrepo import fetch_ucirepo

# ---------------------------------------------------------
# 1. AUTENTICAÇÃO OFICIAL (NUVEM PRIORLABS)
# ---------------------------------------------------------
API_TOKEN = "tabpfn_sk_PWsEO-SmpR1i50K50eiCJkldMKCtz8lmcx__kEbVDis"
tabpfn_client.set_access_token(API_TOKEN)

print("Autenticação configurada com sucesso na nuvem TabPFN.")

# ---------------------------------------------------------
# 2. FETCH & PREP (Base de Dados)
# ---------------------------------------------------------
print("Baixando dados da UCI...")
mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy().fillna(mi_data.data.targets.mode().iloc[0])

# Garantindo que todas as colunas sejam strings (evita erros de serialização)
X_full.columns = X_full.columns.astype(str)

# Split inicial (preservando a integridade do paciente)
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

# Adicionalmente, separamos validação para otimização de limiar
X_train_base, X_val_base, y_train_base, y_val_base = train_test_split(
    X_train_base, y_train_base, test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 3. DAG (Carregando a Matriz do Kaggle)
# ---------------------------------------------------------
print("Carregando matriz de adjacência causal salva...")
df_adj = pd.read_csv("/kaggle/input/datasets/joaoogabriel/adj-matrix/causal_adj_matrix.csv", index_col=0)

targets = y_full.columns.tolist()
binary_targets = targets[:-1]

# Filtro de tempo: garante uso exclusivo de causadores do "Dia 0" (Admissão)
day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
future_cols = day_1_cols + day_2_cols + day_3_cols + targets
adm_cols = [c for c in X_full.columns if c not in future_cols]

# Reconstrói o dicionário de regras (Markov Blanket) a partir do CSV
causal_parents_dict = {}
for target in targets:
    if target in df_adj.columns:
        all_parents = df_adj.index[df_adj[target] == 1].tolist()
        causal_parents = [p for p in all_parents if p in adm_cols]
        causal_parents_dict[target] = causal_parents
    else:
        causal_parents_dict[target] = []

# ---------------------------------------------------------
# 4. PREVISÃO DAG-INFORMED OTIMIZADA (TabPFN CLOUD)
# ---------------------------------------------------------
print("Iniciando treinamento guiado por causalidade via API...")

# Grade de busca de alta precisão (de 0.001 até 0.949)
thresholds = np.arange(0.0001, 0.5, 0.0001)

for target in targets:
    causal_features = causal_parents_dict.get(target, [])
    
    # Seleção de features baseada no Grafo Causal
    X_train_f = X_train_base[causal_features] if causal_features else X_train_base
    X_val_f = X_val_base[causal_features] if causal_features else X_val_base
    X_test_f = X_test_base[causal_features] if causal_features else X_test_base
        
    model = TabPFNClassifier()
    model.fit(X_train_f, y_train_base[target].astype(int))
    
    if target in binary_targets:
        probs = model.predict_proba(X_val_f)[:, 1]
        best_thresh = 0.50
        best_score = 0.0
        
        y_val_t = y_val_base[target].astype(int).values
        
        # Busca pelo limiar ótimo utilizando F2-Score
        for thresh in thresholds:
            preds = (probs >= thresh).astype(int)
            # beta=2.0 confere peso duplo ao Recall, penalizando Falsos Positivos simultaneamente
            score = fbeta_score(y_val_t, preds, beta=2.0, zero_division=0)
            
            if score > best_score:
                best_score = score
                best_thresh = thresh
                
        final_thresh = best_thresh if best_score > 0 else 0.50
        
        final_preds = (model.predict_proba(X_test_f)[:, 1] >= final_thresh).astype(int)
        test_rec = recall_score(y_test_base[target].astype(int), final_preds, zero_division=0)
        test_acc = accuracy_score(y_test_base[target].astype(int), final_preds)
        
        print(f"\n--- {target} ---")
        print(f"Features Causais: {len(causal_features)} | Limiar Ótimo: {final_thresh:.3f}")
        print(f"Recall: {test_rec:.4f} | Acurácia: {test_acc:.4f} | F2-Score (Val): {best_score:.4f}")
        print(confusion_matrix(y_test_base[target].astype(int), final_preds))
        
    else:
        preds = model.predict(X_test_f)
        print(f"\n--- {target} (Acurácia multiclasse: {accuracy_score(y_test_base[target], preds):.4f}) ---")

In [ ]:
import tabpfn_client
from tabpfn_client import TabPFNClassifier
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, accuracy_score, confusion_matrix, f1_score
from ucimlrepo import fetch_ucirepo

# ---------------------------------------------------------
# 1. AUTENTICAÇÃO OFICIAL (NUVEM PRIORLABS)
# ---------------------------------------------------------
API_TOKEN = "tabpfn_sk_PWsEO-SmpR1i50K50eiCJkldMKCtz8lmcx__kEbVDis"
tabpfn_client.set_access_token(API_TOKEN)

print("Autenticação configurada com sucesso na nuvem TabPFN.")

# ---------------------------------------------------------
# 2. FETCH & PREP (Base de Dados)
# ---------------------------------------------------------
print("Baixando dados da UCI...")
mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy().fillna(mi_data.data.targets.mode().iloc[0])

# Garantindo nomenclatura textual para evitar erros de transmissão
X_full.columns = X_full.columns.astype(str)

# Separação estrita dos blocos temporais do dataset
targets = y_full.columns.tolist()
binary_targets = targets[:-1]

day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
future_cols = day_1_cols + day_2_cols + day_3_cols + targets

# Filtrando APENAS as colunas que o médico tem em mãos na ADMISSÃO (Dia 0)
adm_cols = [c for c in X_full.columns if c not in future_cols]
print(f"Total de features livres da Admissão utilizadas: {len(adm_cols)}")

# Split de Pacientes (Evitando vazamento)
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_full[adm_cols], y_full, test_size=0.2, random_state=42
)

# Validação para ajuste fino do limiar de decisão
X_train_base, X_val_base, y_train_base, y_val_base = train_test_split(
    X_train_base, y_train_base, test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 3. PREVISÃO TABPFN LIVRE (SEM FILTRO CAUSAL)
# ---------------------------------------------------------
print("\nIniciando treinamento do TabPFN Livre (Todas as Features de Admissão)...")

thresholds = np.arange(0.001, 0.95, 0.001)

for target in targets:
    # O modelo agora recebe a base completa de admissão (X_train_base contém todas as adm_cols)
    model = TabPFNClassifier()
    model.fit(X_train_base, y_train_base[target].astype(int))
    
    if target in binary_targets:
        probs = model.predict_proba(X_val_base)[:, 1]
        best_thresh = 0.50
        best_score = 0.0
        
        y_val_t = y_val_base[target].astype(int).values
        
        # Otimização via F1-Score para equilibrar Falsos Positivos e Falsos Negativos
        for thresh in thresholds:
            preds = (probs >= thresh).astype(int)
            score = f1_score(y_val_t, preds, zero_division=0)
            
            if score > best_score:
                best_score = score
                best_thresh = thresh
                
        final_thresh = best_thresh if best_score > 0 else 0.50
        
        # Predição na base de teste inédita
        final_probs = model.predict_proba(X_test_base)[:, 1]
        final_preds = (final_probs >= final_thresh).astype(int)
        
        test_rec = recall_score(y_test_base[target].astype(int), final_preds, zero_division=0)
        test_acc = accuracy_score(y_test_base[target].astype(int), final_preds)
        
        print(f"\n==================================================")
        print(f"TARGET: {target} (MODO: LIVRE)")
        print(f"==================================================")
        print(f"Limiar Escolhido: {final_thresh:.3f} | F1-Score (Val): {best_score:.4f}")
        print(f"Recall (Teste): {test_rec:.4f} | Acurácia (Teste): {test_acc:.4f}")
        print("Matriz de Confusão:")
        print(confusion_matrix(y_test_base[target].astype(int), final_preds))
        
    else:
        # Target Multiclasse (LET_IS)
        preds = model.predict(X_test_base)
        print(f"\n==================================================")
        print(f"TARGET MULTICLASSE: {target} (MODO: LIVRE)")
        print(f"==================================================")
        print(f"Acurácia Multiclasse: {accuracy_score(y_test_base[target], preds):.4f}")